## read data

In [3]:
import pandas as pd

# Load your Parquet file
df = pd.read_parquet("../data/clean/ilc_di12_gini.parquet")

# Take the first column (string with comma-separated codes)
first_col = df.columns[0]

# Split into separate columns
split_cols = df[first_col].str.split(",", expand=True)

# Give them names (adjust as needed)
split_cols.columns = ["freq", "age", "unit", "geo"]  # or whatever labels you need

# Drop the original combined column and join the new ones
df = pd.concat([split_cols, df.drop(columns=[first_col])], axis=1)


df_totals = df[df["age"] == ("TOTAL")]

year_cols = [c for c in df_totals.columns if c.strip().isdigit()]
#for col in year_cols:
#    df_totals[col] = df_totals[col].astype(str).str.extract(r"([\d.]+)")[0].astype(float)


for col in year_cols:
    df_totals.loc[:, col] = (
        df_totals[col]
        .astype(str)                # make everything string
        .str.replace(":", "", regex=False)  # drop ":" (missing values)
        .str.extract(r"([\d.]+)")[0]       # keep only number part like 28.5
        .astype(float)                     # to float
    )

df_totals.head()

,freq,age,unit,geo,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,TOTAL,GINI_HND,AL,NaN,NaN,NaN,36.8,35.4,34.3,33.2,33.0,31.0,30.2,NaN,NaN
1,A,TOTAL,GINI_HND,AT,27.6,27.2,27.2,27.9,26.8,27.5,27.0,26.7,27.8,28.1,28.4,NaN
2,A,TOTAL,GINI_HND,BE,25.9,26.2,26.3,26.1,25.7,25.1,25.3,24.1,24.7,24.3,24.6,23.4
3,A,TOTAL,GINI_HND,BG,35.4,37.0,37.7,40.2,39.6,40.8,40.0,39.7,38.4,37.2,38.4,37.7
4,A,TOTAL,GINI_HND,CH,29.5,29.6,29.4,30.1,29.7,30.6,31.2,31.4,31.0,31.5,31.0,NaN


In [4]:
year_cols = [c for c in df_totals.columns if c.strip().isdigit()]

df_long = df_totals.melt(
    id_vars=["freq", "age", "unit", "geo"],
    value_vars=year_cols,
    var_name="year",
    value_name="value"
)

df_long["year"] = df_long["year"].astype(int)
df_long = df_long.dropna(subset=["value"])

In [6]:
## always the same
df_long = df_long.drop(columns=["freq", "age"])

## renaming columns
df_long = df_long.rename(columns={"geo": "country", "unit": "indicator"})

## column order 
df_long = df_long[["country", "indicator", "year", "value"]]

df_long.to_parquet("../data/processed/stg_gini.parquet", index=False)

KeyError: "['freq', 'age'] not found in axis"